# 03. CRAG & Self-RAG: 자기 교정 검색 증강 생성

> CRAG는 **검색 결과**를, Self-RAG는 **생성 결과**를 검증해요. 두 자기 교정 RAG의 차이를 같은 데이터셋 위에서 비교해 봐요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. **CRAG(Corrective RAG)** 패턴을 구현하여 문서 관련성을 평가하고 웹 검색으로 보강하는 RAG 시스템을 만들 수 있어요
2. **GradeDocuments** Pydantic 모델과 `with_structured_output()`으로 이진(yes/no) 문서 평가기를 구성할 수 있어요
3. **Query Rewriter**로 의미적 의도를 추출하여 벡터 검색 최적화 질문을 생성할 수 있어요
4. **Self-RAG** 패턴을 구현하여 문서 관련성 → 환각 검증 → 답변 관련성의 4단계 평가를 수행할 수 있어요
5. `GraphRecursionError`를 처리하여 무한 루프를 안전하게 방어할 수 있어요

## 사전 지식

- 이전 노트북 `02-Agentic-RAG.ipynb`에서 배운 Retriever, RAG Chain, GradeDocuments 패턴
- LangGraph의 StateGraph, 조건부 엣지, 노드 정의 방법
- Pydantic BaseModel과 `with_structured_output()` 기본 사용법

## Part 1: CRAG - Corrective RAG

**CRAG(Corrective-RAG)** 는 검색된 문서를 평가하고, 관련 없는 경우 쿼리를 재작성하여 웹 검색으로 보강하는 전략이에요.

### CRAG의 핵심 아이디어

전통적인 RAG는 검색 결과의 품질에 관계없이 그대로 사용해요. CRAG는 여기에 **자기 교정(Self-Correction)** 단계를 추가해요:

1. 문서를 검색하고 각 문서의 관련성을 평가해요
2. 관련 문서가 하나도 없으면 → 쿼리 재작성 → 웹 검색
3. 관련 문서가 있으면 → 그대로 답변 생성

> 🔑 **핵심 개념**: CRAG는 **논문 작성 과정**과 비슷해요. 참고문헌을 검색했는데 관련 없는 논문만 나왔다면, 검색어를 바꿔서 다른 데이터베이스(Google Scholar, arXiv)에서 다시 찾아보잖아요? CRAG가 정확히 이 과정을 자동화해요 — "검색이 실패했을 때 어떻게 할 것인가?"가 핵심이에요.

### 왜 이전 노트북의 Agentic RAG와 다른가요?

| 비교 항목 | 이전 Agentic RAG (02) | CRAG (이 노트북) |
|----------|----------------------|-----------------|
| 검색 실패 대응 | 쿼리 재작성 → 같은 소스 재검색 | 쿼리 재작성 → **다른 소스(웹)** 검색 |
| 문서 평가 | 전체 문서 일괄 평가 | **개별 문서 하나씩** 관련성 평가 |
| 필터링 | 있음 | 관련 없는 문서 제거 후 나머지만 사용 |
| 루프 구조 | 루프 가능 (무한 위험) | **1회 보완 후 진행** (루프 없음) |

### CRAG 아키텍처

```mermaid
flowchart TD
    A([START]) --> B[retrieve<br/>문서 검색]
    B --> C[grade_documents<br/>관련성 평가]
    C --> D{decide_to_generate<br/>평가 결과?}
    D -- "관련 문서 있음" --> G[generate<br/>답변 생성]
    D -- "관련 문서 없음" --> E[query_rewrite<br/>쿼리 재작성]
    E --> F[web_search_node<br/>웹 검색]
    F --> G
    G --> H([END])

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef decision fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef output fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef terminal fill:#f8d7da,stroke:#dc3545,color:#721c24

    class A,H terminal
    class B,C,E,F process
    class D decision
    class G output
```

### CRAG의 핵심 구성 요소

| 구성 요소 | 역할 | 출력 |
|----------|------|------|
| `retrieve` | 벡터 검색으로 관련 문서 찾기 | 문서 리스트 |
| `grade_documents` | 각 문서의 관련성 평가 | 필터링된 문서 + web_search 플래그 |
| `query_rewrite` | 의미적 의도 추출, 검색 최적화 | 개선된 질문 |
| `web_search_node` | 타빌리(Tavily) 웹 검색 | 보강된 문서 |
| `generate` | 최종 답변 생성 | 답변 텍스트 |

## 1. 환경 설정

In [ ]:
from dotenv import load_dotenv

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# ---------------------------------------------------
# LangSmith 추적 설정
# ---------------------------------------------------
# LangSmith를 사용하면 그래프 실행 과정을 시각적으로 추적할 수 있어요
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-RAG-CRAG-Self-RAG"

## 2. PDF 기반 Retriever 및 RAG Chain 구성

CRAG에서는 Retriever와 생성 체인을 분리해요. 이렇게 하면 각 노드에서 독립적으로 검색과 생성을 제어할 수 있어요.

> 💡 **실무 팁**: `PDFRetrievalChain` 같은 헬퍼 없이 직접 구성하면 내부 동작을 완전히 제어할 수 있어요. FAISS 인덱스, 청킹 전략, 임베딩 모델을 직접 선택할 수 있죠.

**실습 문서**: 소프트웨어정책연구소(SPRi) AI Brief 2023년 12월호  
파일명: `data/SPRI_AI_Brief_2023년12월호_F.pdf`

In [ ]:
import os

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 모델: gpt-4o-mini (비용 효율)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. CRAG 핵심 컴포넌트 구성

### 3-1. 문서 관련성 평가기 (Retrieval Grader)

**GradeDocuments** 모델은 이진(yes/no) 점수로 각 문서의 관련성을 평가해요.

> 🔑 **핵심 개념**: `with_structured_output(GradeDocuments)`를 사용하면 LLM이 자유 텍스트 대신 정해진 구조로 응답해요. `binary_score`가 항상 `"yes"` 또는 `"no"`로만 나와요.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 테스트: 질문과 문서 관련성 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 3-2. 쿼리 재작성기 (Query Rewriter)

검색 결과가 부적절할 때, 원래 질문의 의미적 의도를 파악하여 더 나은 검색어로 변환해요.

> 💡 **실무 팁**: 쿼리 재작성은 벡터 검색 최적화에도 활용돼요. 사용자의 자연어 질문보다 핵심 키워드 중심의 질문이 임베딩 유사도 검색에서 더 좋은 결과를 가져오는 경우가 많아요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 테스트: 질문 재작성 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 3-3. 웹 검색 도구

Tavily Search API를 사용하여 실시간 웹 검색을 수행해요. `TAVILY_API_KEY`가 `.env`에 설정되어 있어야 해요.

> ⚠️ **자주 하는 실수**: `TavilySearchResults`는 도구로 사용할 때와 직접 호출할 때 입력 형식이 달라요. 직접 호출 시에는 `{"query": "..."}`로 입력해요.

In [ ]:
from langchain_tavily import TavilySearch

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 테스트: 웹 검색 결과 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. CRAG 그래프 구성

### 4-1. 상태(State) 정의

CRAG에서는 `web_search` 플래그를 상태에 포함하여 웹 검색 필요 여부를 노드 간에 전달해요.

In [ ]:
from typing import Annotated, List
from typing_extensions import TypedDict

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 4-2. 노드 함수 정의

In [ ]:
from langchain_core.documents import Document

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 4-3. 조건부 엣지 함수

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 4-4. 그래프 조립 및 시각화

> 💡 **핵심 정리**: 그래프 시각화를 보면서 CRAG의 두 경로를 설명해요:
> - **빠른 경로**: retrieve → grade → generate (관련 문서 있을 때)
> - **보완 경로**: retrieve → grade → rewrite → web_search → generate (관련 문서 없을 때)

In [ ]:
from langgraph.graph import END, StateGraph, START

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → retrieve → grade_documents → (generate | query_rewrite → web_search_node → generate) → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. CRAG 실행 테스트

두 가지 경우를 테스트해요:
1. **PDF 내 정보** → 빠른 경로 (retrieve → grade → generate)
2. **PDF에 없는 정보** → 보완 경로 (rewrite → web_search → generate)

In [ ]:
import uuid
from langchain_core.runnables import RunnableConfig

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 테스트 1: PDF에 있는 정보 질문 (빠른 경로)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 테스트 2: PDF에 없는 정보 질문 (보완 경로 - 웹 검색)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## Part 2: Self-RAG - 자기 반성 RAG

**Self-RAG**는 CRAG에서 한 단계 더 나아가, 생성된 답변 자체의 품질도 검증해요. "좋은 답변인가?"를 스스로 평가하고 아니면 다시 시도해요.

> 🔑 **핵심 개념**: CRAG가 **참고자료를 점검하는 사서**라면, Self-RAG는 **자기 답안을 교정하는 수험생**이에요. 시험 답안을 작성한 후 "이 답이 문제지에 있는 자료에 근거하고 있나?", "이 답이 실제로 문제를 해결하나?"를 스스로 되짚어보고, 아니면 다시 작성해요.

### 왜 CRAG만으로는 부족한가요?

CRAG는 **검색 단계**만 검증해요. 하지만 관련 문서를 찾았더라도 LLM이 **엉뚱한 답변**을 생성할 수 있어요:

| 상황 | CRAG의 한계 | Self-RAG의 해결 |
|------|-----------|----------------|
| 관련 문서는 찾았지만 LLM이 문서와 다른 내용을 생성 | 환각 감지 불가 | **Hallucination Grader**로 문서 근거 검증 |
| 관련 문서로 답변했지만 질문의 핵심을 빗나감 | 답변 품질 미검증 | **Answer Grader**로 질문 해결 여부 검증 |
| 웹 검색 없이 문서만 사용 | 문서에 없으면 포기 | 질문 재작성 → 재검색 루프 |

### Self-RAG vs CRAG 비교

| 기능 | CRAG | Self-RAG |
|------|------|----------|
| 문서 관련성 평가 | O | O |
| 쿼리 재작성 | O (웹검색 전) | O (문서 없을 때) |
| 웹 검색 보강 | O | X |
| 환각(Hallucination) 검증 | X | O |
| 답변 관련성 검증 | X | O |
| 자기 루프 | 1회 | 반복 가능 |

### Self-RAG의 4단계 평가

> 💡 **핵심 정리**: Self-RAG는 논문([arXiv:2310.11511](https://arxiv.org/abs/2310.11511))에서 제안된 전략으로, 생성 단계에서 두 가지 질문을 추가로 물어요:
> 1. **환각 검증**: "이 답변이 검색된 문서에 근거하고 있나?"
> 2. **관련성 검증**: "이 답변이 실제로 질문에 답하고 있나?"

### Self-RAG 아키텍처

```mermaid
flowchart TD
    A([START]) --> B[retrieve<br/>문서 검색]
    B --> C[grade_documents<br/>문서 관련성 평가]
    C --> D{decide_to_generate}
    D -- "관련 문서 있음" --> E[generate<br/>답변 생성]
    D -- "관련 문서 없음" --> F[transform_query<br/>질문 재작성]
    F --> B
    E --> G{grade_generation<br/>생성 품질 평가}
    G -- "환각 감지" --> E
    G -- "질문 미해결" --> F
    G -- "좋은 답변" --> H([END])

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef decision fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef output fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef terminal fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef warning fill:#f8d7da,stroke:#dc3545,color:#721c24

    class A,H terminal
    class B,C,E,F process
    class D,G decision
```

> ⚠️ **자주 하는 실수**: Self-RAG에서 루프가 반복되면 `GraphRecursionError`가 발생해요. 반드시 `recursion_limit`을 설정하고 에러를 처리해야 해요. CRAG와 달리 웹 검색 탈출구가 없으므로, 문서에 없는 정보를 질문하면 루프가 끝나지 않을 수 있어요.

## 6. Self-RAG 추가 평가기 구성

Self-RAG는 CRAG의 `retrieval_grader`에 더하여 두 개의 평가기를 더 사용해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 7. Self-RAG 그래프 구성

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → retrieve → grade_documents → (generate | transform_query) → grade_generation → (END | generate | transform_query)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 8. Self-RAG 실행 테스트

> 💡 **핵심 정리**: Self-RAG를 실행하면서 평가 결과에 따라 어떤 경로를 취하는지 로그를 확인해요. 특히 환각 검증에서 `yes`가 나왔을 때와 아닐 때의 차이를 설명해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 9. GraphRecursionError 처리

Self-RAG는 루프 구조를 갖고 있어서 문서에 없는 정보를 질문하면 무한 루프에 빠질 수 있어요. `GraphRecursionError`를 처리하여 안전하게 방어해요.

> ⚠️ **자주 하는 실수**: `recursion_limit`을 설정하지 않으면 기본값(25)까지 반복하다가 에러가 발생해요. 실제 서비스에서는 이 에러를 사용자 친화적인 메시지로 변환해야 해요.

In [ ]:
from langgraph.errors import GraphRecursionError

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 10. 실습: CRAG vs Self-RAG 비교

동일한 질문으로 두 방식을 비교해볼게요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: CRAG와 Self-RAG 비교 실험
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **CRAG(Corrective RAG)**: 문서 관련성 평가 후, 관련 없으면 쿼리 재작성 → 웹 검색으로 보강하는 전략이에요. `GradeDocuments` + `query_rewrite` + `web_search` 세 가지 컴포넌트가 핵심이에요.
- **GradeDocuments**: `Pydantic BaseModel` + `with_structured_output()`으로 LLM이 `binary_score: 'yes'/'no'`를 반환하도록 강제해요.
- **Self-RAG**: CRAG에 환각 검증(Hallucination Grader)과 답변 관련성 검증(Answer Grader)을 추가한 전략이에요. 생성 품질이 낮으면 재생성 또는 질문 재작성 루프를 실행해요.
- **Query Rewriter**: 원본 질문의 의미적 의도를 추출하여 검색에 최적화된 질문으로 변환해요.
- **GraphRecursionError**: `recursion_limit` + `try/except GraphRecursionError`로 무한 루프를 안전하게 방어해요.

### CRAG vs Self-RAG: 언제 어떤 것을 쓸까?

| 상황 | 추천 방식 | 이유 |
|------|----------|------|
| PDF + 웹 검색 모두 가능 | **CRAG** | 웹 검색 폴백으로 답변 범위가 넓어요 |
| 오프라인 환경 (웹 불가) | **Self-RAG** | 웹 검색 없이 자체 검증으로 품질 보장 |
| 답변 정확도가 매우 중요 | **Self-RAG** | 환각+관련성 이중 검증으로 높은 신뢰도 |
| 빠른 응답이 필요 | **CRAG** | 1회 보완으로 끝나서 더 빠름 |
| 가장 강력한 시스템 | **CRAG + Self-RAG 결합** | 다음 노트북의 Adaptive RAG가 이 조합이에요 |

> 💡 **실무 팁**: 실제 프로덕션에서는 CRAG와 Self-RAG를 결합하는 경우가 많아요. "검색 실패 → 웹 보완(CRAG)" + "생성 품질 검증(Self-RAG)"을 합치면 가장 견고한 RAG 시스템이 돼요. 다음 노트북의 **Adaptive RAG**가 바로 이 조합이에요.

## 다음 노트북 예고

다음 `04-Adaptive-RAG.ipynb`에서는 **Query Router와 3단계 조건부 라우팅**을 배워요. CRAG/Self-RAG는 항상 벡터스토어부터 검색하지만, Adaptive RAG는 **질문을 먼저 분류해서 vectorstore / web_search 중 최적 경로를 선택**해요. "2024년 노벨상 수상자" 같은 최신 질문은 처음부터 웹으로, "논문 내용"은 처음부터 벡터스토어로 보내서 시간과 비용을 아끼는 전략이에요.